In [1]:
import os
import glob
import csv
import numpy as np
import scipy.spatial as spatial
from sklearn.decomposition import PCA
from collections import deque

In [2]:
def load_obj_vertices(filepath):
    """Extract vertex coordinates from an OBJ file."""
    vertices = []
    with open(filepath, 'r') as f:
        for line in f:
            if line.startswith('v '):
                parts = line.strip().split()
                if len(parts) >= 4:
                    vertices.append([float(p) for p in parts[1:4]])
    return np.array(vertices)

def point_density(points, k=8):
    """
    6.1 Evaluate point density.
    Returns: mean_neighbor_distance, uniformity_score, density_qualitative
    """
    if len(points) < 2:
        return None, None, "insufficient points"
    kdtree = spatial.cKDTree(points)
    dists, _ = kdtree.query(points, k=k+1)
    mean_dist = np.mean(dists[:, 1:])
    uniformity = np.std(dists[:, 1:]) / mean_dist if mean_dist > 0 else 0
    bbox_size = np.ptp(points, axis=0)
    avg_spacing = mean_dist
    diag = np.linalg.norm(bbox_size)
    if diag == 0:
        density_qual = "single point"
    else:
        rel_spacing = avg_spacing / diag
        if rel_spacing < 0.01:
            density_qual = "high"
        elif rel_spacing > 0.05:
            density_qual = "low"
        else:
            density_qual = "medium"
    return mean_dist, uniformity, density_qual

def shape_distribution(points):
    """
    6.2 Determine shape (linear/planar/volumetric) using PCA.
    Returns: elongation_ratio, dominant_direction (eigenvector), shape_type
    """
    pca = PCA(n_components=3)
    pca.fit(points)
    evals = pca.explained_variance_
    if len(evals) < 3:
        evals = np.pad(evals, (0, 3-len(evals)), constant_values=0)
    elongation = evals[0] / (evals[2] + 1e-9)
    evals_norm = evals / (evals.sum() + 1e-9)
    if evals_norm[0] > 0.8:
        shape = "linear"
    elif evals_norm[1] > 0.3:
        shape = "planar"
    else:
        shape = "volumetric"
    # dominant direction, first principal component
    dom_dir = pca.components_[0]
    return elongation, dom_dir, shape

def surface_structure(points, k=8):
    """
    6.3 Estimate curvature and surface character.
    Returns: mean_curvature, curvature_std, classification
    """
    if len(points) < k:
        return None, None, "too few points"
    kdtree = spatial.cKDTree(points)
    curvatures = []
    for i, p in enumerate(points):
        idx = kdtree.query(p, k=k+1)[1][1:]
        if len(idx) < 3:
            continue
        neigh = points[idx]
        centered = neigh - np.mean(neigh, axis=0)
        cov = (centered.T @ centered) / len(neigh)
        evals = np.linalg.eigvalsh(cov)
        evals = np.sort(evals)[::-1]
        curv = evals[2] / (evals.sum() + 1e-9)
        curvatures.append(curv)
    curvatures = np.array(curvatures)
    mean_curv = np.mean(curvatures)
    std_curv = np.std(curvatures)
    if mean_curv < 0.05:
        cls = "smooth"
    elif mean_curv < 0.15:
        cls = "slightly curved"
    elif mean_curv < 0.35:
        cls = "highly curved"
    else:
        cls = "complex (combined)"
    return mean_curv, std_curv, cls

def normals_consistency(points, k=8):
    """
    6.4 Estimate normals via PCA and evaluate their consistency.
    Returns: mean_angular_consistency, consistency_qualitative
    """
    if len(points) < k:
        return None, "insufficient points"
    kdtree = spatial.cKDTree(points)
    normals = []
    for i, p in enumerate(points):
        idx = kdtree.query(p, k=k+1)[1][1:]
        if len(idx) < 3:
            continue
        neigh = points[idx]
        centered = neigh - np.mean(neigh, axis=0)
        cov = (centered.T @ centered) / len(neigh)
        evals, evecs = np.linalg.eigh(cov)
        normal = evecs[:, 0]
        normals.append(normal)
    normals = np.array(normals)
    cons_angles = []
    for i, n in enumerate(normals):
        idx = kdtree.query(points[i], k=k+1)[1][1:]
        for j in idx:
            if j < len(normals):
                dot = np.abs(np.dot(n, normals[j]))
                ang = np.arccos(np.clip(dot, -1, 1))
                cons_angles.append(ang)
    if not cons_angles:
        return None, "no data"
    mean_ang = np.mean(cons_angles)
    if mean_ang < 0.2:       # ~11 degrees
        qual = "high (planar/smooth)"
    elif mean_ang < 0.5:     # ~28 degrees
        qual = "medium (cylindrical/spherical)"
    else:
        qual = "low (complex geometry or noise)"
    return mean_ang, qual

def topological_connectivity(points, dist_threshold=None):
    """
    6.5 Evaluate connectivity, components, holes.
    Build graph using distance threshold (estimate from average spacing).
    Returns: n_components, isolated_clusters, integrity_estimate
    """
    if len(points) < 2:
        return 0, [], "no points"
    # estimate threshold = 2.5 x average_nearest_neighbor_distance
    kdtree = spatial.cKDTree(points)
    if dist_threshold is None:
        dists, _ = kdtree.query(points, k=2)
        avg_nn_dist = np.mean(dists[:, 1])
        dist_threshold = avg_nn_dist * 2.5

    adjacency = {}
    for i, p in enumerate(points):
        neighbors = kdtree.query_ball_point(p, dist_threshold)
        adjacency[i] = [j for j in neighbors if j != i]

    visited = set()
    components = []
    for i in range(len(points)):
        if i not in visited:
            q = deque([i])
            comp = []
            while q:
                cur = q.popleft()
                if cur in visited:
                    continue
                visited.add(cur)
                comp.append(cur)
                for nb in adjacency.get(cur, []):
                    if nb not in visited:
                        q.append(nb)
            components.append(comp)
    n_components = len(components)
    # isolated clusters are components with size <= 3
    isolated = [c for c in components if len(c) <= 3]
    # integrity: ratio of largest component size to total points
    if len(points) > 0:
        largest = max(len(c) for c in components) if components else 0
        integrity = largest / len(points)
    else:
        integrity = 0
    if integrity > 0.99:
        integ_qual = "highly intact (one main component)"
    elif integrity > 0.8:
        integ_qual = "mostly intact with small outliers"
    else:
        integ_qual = "fragmented or many components"
    return n_components, isolated, integ_qual

def analyze_segment(points):
    if len(points) == 0:
        return "No points in segment."
    print(f"Analyzing segment with {len(points)} points.\n")
    # 6.1 Density
    mean_dist, uniformity, dens_qual = point_density(points)
    print(f"6.1 Point density:")
    print(f"    Mean neighbor distance: {mean_dist:.4f}")
    print(f"    Uniformity (lower is better): {uniformity:.3f}")
    print(f"    Qualitative density: {dens_qual}\n")
    # 6.2 Shape
    elong, dom_dir, shape = shape_distribution(points)
    print(f"6.2 Shape distribution:")
    print(f"    Elongation (λ1/λ3): {elong:.2f}")
    print(f"    Dominant direction: {dom_dir}")
    print(f"    Shape type: {shape}\n")
    # 6.3 Surface structure
    mean_curv, std_curv, surf_class = surface_structure(points)
    print(f"6.3 Surface structure:")
    print(f"    Mean curvature (surface variation): {mean_curv:.4f}")
    print(f"    Std deviation: {std_curv:.4f}")
    print(f"    Classification: {surf_class}\n")
    # 6.4 Normal consistency
    mean_ang, cons_qual = normals_consistency(points)
    if mean_ang is not None:
        print(f"6.4 Normal consistency:")
        print(f"    Mean angle between neighboring normals: {np.degrees(mean_ang):.1f}°")
        print(f"    Consistency: {cons_qual}\n")
    else:
        print("6.4 Normal consistency: insufficient data.\n")
    # 6.5 Topological connectivity
    n_comp, isolated, integ = topological_connectivity(points)
    print(f"6.5 Topological connectivity:")
    print(f"    Number of connected components: {n_comp}")
    print(f"    Isolated clusters (size ≤ 3): {len(isolated)}")
    print(f"    Integrity: {integ}\n")
    # 6.6 Summary
    print("6.6 Analysis summary:")
    print(f"    - Density: {dens_qual}, neighbor distance {mean_dist:.4f}")
    print(f"    - Shape: {shape} (elongation {elong:.2f})")
    print(f"    - Surface: {surf_class} (curvature {mean_curv:.3f})")
    print(f"    - Normals: {cons_qual if mean_ang else 'N/A'}")
    print(f"    - Connectivity: {integ}")

In [3]:
def analyze_all_in_directory(directory_path, output_csv="segments_summary.csv"):
    """
    Processes all .obj files in directory_path.
    Performs detailed analysis for each and saves brief results to CSV.
    """
    obj_files = glob.glob(os.path.join(directory_path, "*.obj"))
    if not obj_files:
        print(f"No .obj files found in {directory_path}.")
        return

    print(f"Found {len(obj_files)} OBJ files. Starting analysis...\n")
    print("=" * 80)

    csv_rows = []
    csv_headers = ["file", "num_points", "density_qual", "mean_neighbor_dist",
                   "shape_type", "elongation", "surface_class", "mean_curvature",
                   "normals_consistency", "n_components", "integrity_qual"]

    for obj_file in obj_files:
        file_name = os.path.basename(obj_file)
        print(f"\n>>> Processing: {file_name} <<<\n")
        points = load_obj_vertices(obj_file)
        if points.size == 0:
            print(f"Skipping {file_name}: no vertices.\n")
            continue

        mean_dist, uniformity, dens_qual = point_density(points)
        elong, dom_dir, shape_type = shape_distribution(points)
        mean_curv, std_curv, surf_class = surface_structure(points)
        mean_ang, cons_qual = normals_consistency(points)
        n_comp, isolated, integ_qual = topological_connectivity(points)

        csv_rows.append({
            "file": file_name,
            "num_points": len(points),
            "density_qual": dens_qual,
            "mean_neighbor_dist": f"{mean_dist:.4f}" if mean_dist else "N/A",
            "shape_type": shape_type,
            "elongation": f"{elong:.2f}",
            "surface_class": surf_class,
            "mean_curvature": f"{mean_curv:.4f}" if mean_curv else "N/A",
            "normals_consistency": cons_qual if mean_ang is not None else "N/A",
            "n_components": n_comp,
            "integrity_qual": integ_qual
        })

        print(f"\n--- BRIEF SUMMARY FOR {file_name} ---")
        print(f"Points: {len(points)}")
        print(f"Density: {dens_qual} (mean distance {mean_dist:.4f})")
        print(f"Shape: {shape_type} (elongation {elong:.2f})")
        print(f"Surface: {surf_class} (curvature {mean_curv:.3f})")
        print(f"Normals: {cons_qual if mean_ang else 'not evaluated'}")
        print(f"Connectivity: {integ_qual} (components: {n_comp})")
        print("-" * 80)

    with open(output_csv, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=csv_headers)
        writer.writeheader()
        writer.writerows(csv_rows)

    print(f"\nSummary table saved to {output_csv}")

if __name__ == "__main__":
    dataset_path = "dataset_task3"
    analyze_all_in_directory(dataset_path, output_csv="segments_analysis_report.csv")

Found 11 OBJ files. Starting analysis...


>>> Processing: 13.01.obj <<<


--- BRIEF SUMMARY FOR 13.01.obj ---
Points: 1204
Density: medium (mean distance 4.0116)
Shape: planar (elongation 2.12)
Surface: slightly curved (curvature 0.083)
Normals: low (complex geometry or noise)
Connectivity: fragmented or many components (components: 57)
--------------------------------------------------------------------------------

>>> Processing: 13.02.obj <<<


--- BRIEF SUMMARY FOR 13.02.obj ---
Points: 1204
Density: medium (mean distance 4.0176)
Shape: planar (elongation 2.12)
Surface: slightly curved (curvature 0.093)
Normals: low (complex geometry or noise)
Connectivity: mostly intact with small outliers (components: 51)
--------------------------------------------------------------------------------

>>> Processing: 13.03.obj <<<


--- BRIEF SUMMARY FOR 13.03.obj ---
Points: 376
Density: medium (mean distance 4.5409)
Shape: planar (elongation 4.79)
Surface: smooth (curvature 0.014)
Normals: h